In [13]:
import mne
import numpy as np


raw = mne.io.read_raw_fif(r"D:\Signal_processing\EEG_Pre_processing\cleaned_eeg_data_raw.fif", preload=True)

print("DONE")

Opening raw data file D:\Signal_processing\EEG_Pre_processing\cleaned_eeg_data_raw.fif...
    Range : 0 ... 79499 =      0.000 ...   317.996 secs
Ready.
Reading 0 ... 79499  =      0.000 ...   317.996 secs...
DONE


In [14]:
# Extract only EEG channels data (excluding event/stimulus channels)
eeg_picks = mne.pick_types(raw.info, eeg=True, stim=False, exclude=[])
eeg_data = raw.get_data(picks=eeg_picks)
eeg_ch_names = [raw.info['ch_names'][i] for i in eeg_picks]

In [19]:
# Calculate standard deviation for each channel to identify anomalies
channel_stds = np.std(eeg_data, axis=1)

In [20]:
# Define upper and lower statistical thresholds (3-Sigma Rule)
threshold_high = np.mean(channel_stds) + 3 * np.std(channel_stds)
threshold_low = np.mean(channel_stds) - 2 * np.std(channel_stds)

In [17]:
# Identify bad channels
bad_channels = []
for idx, ch_name in enumerate(raw.info['ch_names']):
    if channel_stds[idx] > threshold_high or channel_stds[idx] < threshold_low:
        bad_channels.append(ch_name)

In [21]:
# Identify bad EEG channels
bad_channels = []
for idx, ch_name in enumerate(eeg_ch_names):
    if channel_stds[idx] > threshold_high or channel_stds[idx] < threshold_low:
        bad_channels.append(ch_name)

In [25]:
# Handle bad channels: mark and interpolate using spatial neighbors
if bad_channels:
    print(f"⚠️ Bad EEG channels detected: {bad_channels}")
    raw.info['bads'] = bad_channels
    # Interpolate bad channels using Spherical Spline Interpolation
    raw.interpolate_bads(reset=True, mode='accurate')
    print("✅ Bad EEG channels successfully interpolated!")
else:
    print("✅ No bad or noisy EEG channels detected. All 19 electrodes are clean!")

✅ No bad or noisy EEG channels detected. All 19 electrodes are clean!


In [26]:
# Create fixed-length events every 2.0 seconds
events = mne.make_fixed_length_events(raw, duration=2.0)

In [27]:
# Define peak-to-peak rejection threshold for EEG channels (100 microvolts)
reject_criteria = dict(eeg=100e-6)

In [28]:
# Create Epochs and automatically reject noisy segments
epochs = mne.Epochs(
    raw, 
    events, 
    tmin=0, 
    tmax=2.0, 
    baseline=None, 
    reject=reject_criteria,
    preload=True
)

Not setting metadata
159 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 159 events and 501 original time points ...
    Rejecting  epoch based on EEG : ['F7', 'T3', 'Cz']
    Rejecting  epoch based on EEG : ['T6']
    Rejecting  epoch based on EEG : ['T6']
    Rejecting  epoch based on EEG : ['T6']
    Rejecting  epoch based on EEG : ['T6']
    Rejecting  epoch based on EEG : ['T6']
    Rejecting  epoch based on EEG : ['T6']
    Rejecting  epoch based on EEG : ['T6']
    Rejecting  epoch based on EEG : ['T6']
    Rejecting  epoch based on EEG : ['T6']
11 bad epochs dropped


In [29]:
# Display summary of clean vs total epochs
print(f"✅ Created {len(epochs)} clean epochs out of {len(events)} total segments.")
print(f"📊 Drop log summary: {epochs.drop_log_stats():.2f}% of epochs rejected due to heavy artifacts.")

✅ Created 148 clean epochs out of 159 total segments.
📊 Drop log summary: 6.92% of epochs rejected due to heavy artifacts.
